In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from pathlib import Path

results_dir = Path(".")  # notebook is in results/
df = pd.read_csv(results_dir / "lrt_nb_vs_zinb_results.tsv", sep="\t")

In [ ]:
# Parse comparison column into dataset and model_type
df[["dataset", "model_type"]] = df["comparison"].str.split(" / ", expand=True)

# Dataset display order and labels
dataset_order = ["seelig_cm", "shendure_obs", "shendure_cm"]
dataset_labels = {
    "seelig_cm":    "Seelig (CM)",
    "shendure_obs": "Shendure (obs)",
    "shendure_cm":  "Shendure (CM)",
}

by_cre = df[df["model_type"] == "by_cre"]
by_ct  = df[df["model_type"] == "by_cell_type"]

print("by_cre n per dataset:", by_cre.groupby("dataset").size().to_dict())
print("by_cell_type n per dataset:", by_ct.groupby("dataset").size().to_dict())

In [ ]:
fig, axes = plt.subplots(
    2, 3,
    figsize=(10, 6),
    constrained_layout=True,
)

metric = "delta_aic"  # ΔAIC: positive = ZINB preferred
metric_label = r"$\Delta$AIC  (NB $-$ ZINB, positive = ZINB preferred)"

for col, ds in enumerate(dataset_order):
    # --- top row: by_cre violin ---
    ax = axes[0, col]
    sub = by_cre[by_cre["dataset"] == ds][metric].dropna()
    ax.axhline(0, color="black", lw=0.8, ls="--", zorder=0)
    parts = ax.violinplot(sub, positions=[0], showmedians=True, widths=0.6)
    parts["cmedians"].set_color("firebrick")
    for pc in parts["bodies"]:
        pc.set_facecolor("steelblue")
        pc.set_alpha(0.7)
    ax.set_title(dataset_labels[ds], fontsize=9)
    ax.set_xticks([])
    ax.set_xlim(-0.6, 0.6)
    n = len(sub)
    ax.text(0.97, 0.97, f"n={n}", transform=ax.transAxes,
            ha="right", va="top", fontsize=7, color="gray")
    if col == 0:
        ax.set_ylabel(metric_label, fontsize=8)

    # --- bottom row: by_cell_type strip ---
    ax2 = axes[1, col]
    sub2 = by_ct[by_ct["dataset"] == ds][[metric, "model"]].dropna()
    ax2.axhline(0, color="black", lw=0.8, ls="--", zorder=0)
    jitter = np.random.default_rng(0).uniform(-0.15, 0.15, len(sub2))
    ax2.scatter(jitter, sub2[metric], s=30, color="steelblue", alpha=0.8, zorder=2)
    # label each point with its model name
    for x, (_, row) in zip(jitter, sub2.iterrows()):
        ax2.text(x, row[metric], "  " + row["model"], fontsize=6, va="center", clip_on=True)
    ax2.set_xticks([])
    ax2.set_xlim(-0.6, 0.6)
    n2 = len(sub2)
    ax2.text(0.97, 0.97, f"n={n2}", transform=ax2.transAxes,
             ha="right", va="top", fontsize=7, color="gray")
    if col == 0:
        ax2.set_ylabel(metric_label, fontsize=8)

# Row labels
axes[0, 0].annotate("by CRE", xy=(0, 0.5), xytext=(-0.35, 0.5),
    xycoords="axes fraction", textcoords="axes fraction",
    fontsize=9, fontweight="bold", rotation=90, va="center", ha="center")
axes[1, 0].annotate("by cell type", xy=(0, 0.5), xytext=(-0.35, 0.5),
    xycoords="axes fraction", textcoords="axes fraction",
    fontsize=9, fontweight="bold", rotation=90, va="center", ha="center")

fig.suptitle("LRT: NB vs ZINB — distribution of ΔAIC per model", fontsize=10)

out = results_dir / "lrt_nb_vs_zinb_delta_aic.svg"
fig.savefig(out, format="svg", bbox_inches="tight")
print(f"Saved {out}")
plt.show()